# Seed Dataset Ingestion Pipeline 
Indexes Vietnamese legal documents from `vohuutridung/vietnamese-legal-documents` into:
- Neon Postgres
- Qdrant Cloud

In [ ]:
!pip install -q datasets FlagEmbedding qdrant-client psycopg2-binary pydantic python-dotenv tqdm

In [ ]:
import os
try:
    from google.colab import userdata
    QDRANT_HOST = userdata.get('QDRANT_HOST')
    QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
    DATABASE_DSN = userdata.get('DATABASE_DSN')
QDRANT_COLLECTION = os.getenv('QDRANT_COLLECTION', 'legal_chunks_v1')
print('Qdrant Host:', QDRANT_HOST)
print('Postgres DSN ready:', bool(DATABASE_DSN))

In [ ]:
import re
import uuid
import hashlib
import unicodedata
from typing import List, Dict, Any, Optional
from pydantic import BaseModel

class LegalChunkRecord(BaseModel):
    chunk_id: str
    doc_id: str
    doc_number: str
    doc_title: str
    doc_type: str
    issuing_body: Optional[str] = None
    issued_year: int
    effective_status: str = 'active'
    chapter_id: Optional[str] = None
    section_id: Optional[str] = None
    article_id: str
    clause_id: str = 'ALL'
    breadcrumb: str
    raw_content: str
    token_count: int = 0
    chunk_sha256: str
    referenced_provisions: List[str] = []

    @classmethod
    def generate_chunk_id(cls, doc_number: str, article_id: str, clause_id: str) -> str:
        key = f'{doc_number}_{article_id}_{clause_id}'.strip()
        return str(uuid.uuid5(uuid.NAMESPACE_OID, key))

RE_CHAPTER = re.compile(r'^Chương\s+([IVXLCDM\d]+)[\.\:\s]*(.*)$', re.IGNORECASE)
RE_SECTION = re.compile(r'^Mục\s+(\d+)[\.\:\s]*(.*)$', re.IGNORECASE)
RE_ARTICLE = re.compile(r'^Điều\s+(\d+[a-zđ]?)\.\s*(.*)$', re.IGNORECASE)
RE_CLAUSE = re.compile(r'^(\d+)\.\s+(.*)$')
RE_PROVISION_REF = re.compile(r'\bĐiều\s+\d+[a-zđ]?\b', re.IGNORECASE)

class LegalParserService:
    def __init__(self, tokenizer=None):
        self.tokenizer = tokenizer

    def _count_tokens(self, text: str) -> int:
        if self.tokenizer:
            return len(self.tokenizer.encode(text))
        return int(len(text.split()) * 1.3)

    def _extract_references(self, text: str) -> List[str]:
        return sorted(list(set(RE_PROVISION_REF.findall(text))))

    def _flush_article(self, doc_meta, chapter, section, article_id, article_title, lead_in, clauses):
        chunks = []
        doc_number = doc_meta['doc_number']
        parts = [doc_number]
        if chapter: parts.append(chapter)
        if section: parts.append(section)
        parts.append(f'{article_id}. {article_title}'.strip())
        base_breadcrumb = ' > '.join(parts)

        full_lines = [f'{article_id}. {article_title}'.strip()]
        if lead_in: full_lines.append(lead_in)
        for cl in clauses: full_lines.append(f"{cl['code']}. {cl['text']}")
        full_content = '\n'.join(full_lines).strip()
        total_tokens = self._count_tokens(full_content)

        if total_tokens <= 1024 or not clauses:
            cid = LegalChunkRecord.generate_chunk_id(doc_number, article_id, 'ALL')
            chunks.append(LegalChunkRecord(
                chunk_id=cid, doc_id=doc_meta['doc_id'], doc_number=doc_number,
                doc_title=doc_meta['doc_title'], doc_type=doc_meta['doc_type'],
                issuing_body=doc_meta.get('issuing_body'), issued_year=doc_meta['issued_year'],
                effective_status=doc_meta.get('effective_status', 'active'),
                chapter_id=chapter, section_id=section, article_id=article_id, clause_id='ALL',
                breadcrumb=base_breadcrumb, raw_content=full_content, token_count=total_tokens,
                chunk_sha256=hashlib.sha256(full_content.encode('utf-8')).hexdigest(),
                referenced_provisions=self._extract_references(full_content)
            ))
            return chunks

        for cl in clauses:
            clause_id = f"Khoản {cl['code']}"
            c_content = f"{base_breadcrumb}\n{article_id}. {article_title}\n{lead_in}\n{cl['code']}. {cl['text']}".strip()
            cid = LegalChunkRecord.generate_chunk_id(doc_number, article_id, clause_id)
            chunks.append(LegalChunkRecord(
                chunk_id=cid, doc_id=doc_meta['doc_id'], doc_number=doc_number,
                doc_title=doc_meta['doc_title'], doc_type=doc_meta['doc_type'],
                issuing_body=doc_meta.get('issuing_body'), issued_year=doc_meta['issued_year'],
                effective_status=doc_meta.get('effective_status', 'active'),
                chapter_id=chapter, section_id=section, article_id=article_id, clause_id=clause_id,
                breadcrumb=f"{base_breadcrumb} > {clause_id}", raw_content=c_content,
                token_count=self._count_tokens(c_content),
                chunk_sha256=hashlib.sha256(c_content.encode('utf-8')).hexdigest(),
                referenced_provisions=self._extract_references(c_content)
            ))
        return chunks

    def parse_document(self, text: str, doc_meta: Dict[str, Any]) -> List[LegalChunkRecord]:
        text = unicodedata.normalize('NFC', text)
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        current_chapter, current_section, current_article_id = None, None, None
        current_article_title, article_lead_in, clauses = '', [], []
        current_clause_idx = None
        records = []

        for line in lines:
            m_chap = RE_CHAPTER.match(line)
            if m_chap:
                current_chapter = f"Chương {m_chap.group(1)} {m_chap.group(2)}".strip()
                current_section = None
                continue
            m_sec = RE_SECTION.match(line)
            if m_sec:
                current_section = f"Mục {m_sec.group(1)} {m_sec.group(2)}".strip()
                continue
            m_art = RE_ARTICLE.match(line)
            if m_art:
                if current_article_id:
                    records.extend(self._flush_article(doc_meta, current_chapter, current_section, current_article_id, current_article_title, '\n'.join(article_lead_in), clauses))
                current_article_id = f"Điều {m_art.group(1)}"
                current_article_title = m_art.group(2).strip()
                article_lead_in, clauses, current_clause_idx = [], [], None
                continue
            if current_article_id:
                m_cl = RE_CLAUSE.match(line)
                if m_cl:
                    clauses.append({'code': m_cl.group(1), 'text': m_cl.group(2)})
                    current_clause_idx = len(clauses) - 1
                elif current_clause_idx is not None:
                    clauses[current_clause_idx]['text'] += f'\n{line}'
                else:
                    article_lead_in.append(line)
        if current_article_id:
            records.extend(self._flush_article(doc_meta, current_chapter, current_section, current_article_id, current_article_title, '\n'.join(article_lead_in), clauses))
        return records

In [ ]:
import torch
import psycopg2
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

embed_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=(device == 'cuda'), device=device)
parser = LegalParserService()

qdrant_client = QdrantClient(url=QDRANT_HOST, api_key=QDRANT_API_KEY, timeout=60.0)
pg_conn = psycopg2.connect(DATABASE_DSN)
pg_conn.autocommit = True

In [ ]:
import re
from datasets import load_dataset
from tqdm.auto import tqdm

TARGET_TYPES = {'Bộ luật', 'Luật', 'Nghị định', 'Thông tư'}

print('Streaming metadata and content datasets...')
meta_stream = load_dataset('vohuutridung/vietnamese-legal-documents', 'metadata', split='data', streaming=True)
content_stream = load_dataset('vohuutridung/vietnamese-legal-documents', 'content', split='data', streaming=True)
content_iter = iter(content_stream)

batch_points = []
batch_db_docs = []
batch_db_chunks = []
processed_docs = 0
total_chunks = 0

for m_item in tqdm(meta_stream, desc='Processing legal documents'):
    c_item = next(content_iter)
    assert m_item['id'] == c_item['id'], f"Mismatched id: {m_item['id']} vs {c_item['id']}"

    doc_type = m_item.get('legal_type', '').strip()
    if doc_type not in TARGET_TYPES:
        continue

    raw_text = c_item.get('content', '').strip()
    if not raw_text:
        continue

    iss_date = m_item.get('issuance_date', '')
    m_year = re.search(r'\b(19\d{2}|20\d{2})\b', iss_date)
    issued_year = int(m_year.group(1)) if m_year else 2024

    doc_id = str(uuid.uuid5(uuid.NAMESPACE_OID, str(m_item['id'])))
    doc_number = m_item.get('document_number', f"DOC-{m_item['id']}").strip()
    doc_title = m_item.get('title', '').strip()
    content_sha256 = hashlib.sha256(raw_text.encode('utf-8')).hexdigest()

    meta_dict = {
        'doc_id': doc_id,
        'doc_number': doc_number,
        'doc_title': doc_title,
        'doc_type': doc_type,
        'issuing_body': m_item.get('issuing_authority'),
        'issued_year': issued_year,
        'effective_status': 'active'
    }

    batch_db_docs.append((
        doc_id, doc_number, doc_title, doc_type, m_item.get('issuing_authority'),
        m_item.get('signers'), None, None, None, 'active', m_item.get('legal_sectors'), content_sha256
    ))

    chunks = parser.parse_document(raw_text, meta_dict)
    if not chunks:
        continue

    chunk_texts = [c.raw_content for c in chunks]
    embeddings = embed_model.encode(
        chunk_texts, batch_size=16, max_length=1024,
        return_dense=True, return_sparse=True, return_colbert_vecs=False
    )

    for chk, d_vec, s_dict in zip(chunks, embeddings['dense_vecs'], embeddings['lexical_weights']):
        sparse_indices = [int(k) for k in s_dict.keys()]
        sparse_values = [float(v) for v in s_dict.values()]

        point = models.PointStruct(
            id=chk.chunk_id,
            vector={
                'dense_vector': d_vec.tolist(),
                'sparse_vector': models.SparseVector(indices=sparse_indices, values=sparse_values)
            },
            payload={
                'doc_id': chk.doc_id,
                'doc_number': chk.doc_number,
                'doc_type': chk.doc_type,
                'issued_year': chk.issued_year,
                'effective_status': chk.effective_status,
                'article_code': chk.article_id,
                'clause_code': chk.clause_id,
                'breadcrumb': chk.breadcrumb,
                'chunk_sha256': chk.chunk_sha256
            }
        )
        batch_points.append(point)
        batch_db_chunks.append((
            chk.chunk_id, chk.doc_id, None, chk.breadcrumb,
            chk.raw_content, chk.token_count, chk.chunk_sha256, True
        ))

    processed_docs += 1
    total_chunks += len(chunks)

    if len(batch_points) >= 100:
        qdrant_client.upsert(collection_name=QDRANT_COLLECTION, points=batch_points)
        with pg_conn.cursor() as cur:
            cur.executemany(
                '''INSERT INTO legal_documents (doc_id, doc_number, doc_title, doc_type, issuing_body, signer_name, issued_date, effective_date, expiry_date, effective_status, legal_field, content_sha256)
                   VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                   ON CONFLICT (doc_number) DO NOTHING;''',
                batch_db_docs
            )
            cur.executemany(
                '''INSERT INTO legal_chunks (chunk_id, doc_id, section_id, breadcrumb, raw_content, token_count, chunk_sha256, is_indexed_qdrant)
                   VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                   ON CONFLICT (chunk_id) DO NOTHING;''',
                batch_db_chunks
            )
        batch_points.clear()
        batch_db_docs.clear()
        batch_db_chunks.clear()

if batch_points:
    qdrant_client.upsert(collection_name=QDRANT_COLLECTION, points=batch_points)
    with pg_conn.cursor() as cur:
        cur.executemany(
            '''INSERT INTO legal_documents (doc_id, doc_number, doc_title, doc_type, issuing_body, signer_name, issued_date, effective_date, expiry_date, effective_status, legal_field, content_sha256)
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
               ON CONFLICT (doc_number) DO NOTHING;''',
            batch_db_docs
        )
        cur.executemany(
            '''INSERT INTO legal_chunks (chunk_id, doc_id, section_id, breadcrumb, raw_content, token_count, chunk_sha256, is_indexed_qdrant)
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
               ON CONFLICT (chunk_id) DO NOTHING;''',
            batch_db_chunks
        )

print(f'Successfully indexed {processed_docs} documents and {total_chunks} chunks.')